# Torsion-chain discrete and Gaussian bias

Compare plain MC, a discrete state-counting bias, and a state-conditioned torsional Gaussian bias.


In [ ]:
import matplotlib.pyplot as plt
import metatally as mt


In [ ]:
model = mt.TorsionChain(n_variables=6, radix=3, barrier=4.0)
space = model.state_space()
x0 = model.initial_state()
n_steps = 10_000


In [ ]:
plain = mt.MetropolisSampler(
    model,
    space,
    initial=x0,
    bias=mt.NoBias(),
    step_size=0.55,
    seed=7,
)

discrete = mt.MetropolisSampler(
    model,
    space,
    initial=x0,
    bias=mt.DiscreteStateBias.from_space(space, height=0.02),
    step_size=0.55,
    seed=7,
)

gaussian = mt.MetropolisSampler(
    model,
    space,
    initial=x0,
    bias=mt.StateConditionedTorsionBias.from_space(
        space, height=0.02, sigma=0.25, n_grid=96
    ),
    step_size=0.55,
    seed=7,
)


In [ ]:
for _ in range(n_steps):
    plain.step()
    discrete.step()
    gaussian.step()

for name, sampler in [("plain", plain), ("discrete", discrete), ("gaussian", gaussian)]:
    print(
        f"{name:8s} n_visited={sampler.n_visited:4d} "
        f"coverage={sampler.coverage:.3f} "
        f"acceptance={sampler.acceptance_rate:.3f}"
    )


In [ ]:
plt.figure()
plt.plot(plain.steps, plain.n_unique_by_step, label="plain MC")
plt.plot(discrete.steps, discrete.n_unique_by_step, label="DiscreteStateBias")
plt.plot(gaussian.steps, gaussian.n_unique_by_step, label="StateConditionedTorsionBias")
plt.xlabel("MC step")
plt.ylabel("Unique global states visited")
plt.legend()
plt.tight_layout()


In [ ]:
plt.figure()
plt.plot(plain.steps, plain.n_unique_by_step / space.n_states, label="plain MC")
plt.plot(discrete.steps, discrete.n_unique_by_step / space.n_states, label="DiscreteStateBias")
plt.plot(gaussian.steps, gaussian.n_unique_by_step / space.n_states, label="StateConditionedTorsionBias")
plt.xlabel("MC step")
plt.ylabel("Fraction of states visited")
plt.legend()
plt.tight_layout()
